In [2]:
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

df = fetch_california_housing(as_frame=True).frame

X = df.drop(columns='MedHouseVal')
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

(16512, 8) (4128, 8)


In [3]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

def evaluasi(model, nama):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(f"{nama}")
    print(f"  MAE  : {mean_absolute_error(y_test, pred):.3f}")
    print(f"  RMSE : {root_mean_squared_error(y_test, pred):.3f}")
    print(f"  R2   : {r2_score(y_test, pred):.3f}")

evaluasi(DummyRegressor(strategy="mean"), "Dummy (tebak rata-rata)")
evaluasi(LinearRegression(), "Linear Regression")

Dummy (tebak rata-rata)
  MAE  : 0.906
  RMSE : 1.145
  R2   : -0.000
Linear Regression
  MAE  : 0.533
  RMSE : 0.746
  R2   : 0.576


In [4]:
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

evaluasi(RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1), "Random Forest")
evaluasi(HistGradientBoostingRegressor(random_state=42), "Gradient Boosting")

Random Forest
  MAE  : 0.328
  RMSE : 0.506
  R2   : 0.805
Gradient Boosting
  MAE  : 0.311
  RMSE : 0.465
  R2   : 0.835


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])

evaluasi(pipe_lr, "Pipeline: Scaler + Linear Regression")

Pipeline: Scaler + Linear Regression
  MAE  : 0.533
  RMSE : 0.746
  R2   : 0.576


In [6]:
from sklearn.model_selection import cross_val_score

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting": HistGradientBoostingRegressor(random_state=42),
}

for nama, m in models.items():
    skor = cross_val_score(m, X_train, y_train, cv=5, scoring="neg_mean_absolute_error")
    print(f"{nama}: MAE {-skor.mean():.3f} ± {skor.std():.3f}")

Linear Regression: MAE 0.529 ± 0.005
Random Forest: MAE 0.334 ± 0.005
Gradient Boosting: MAE 0.317 ± 0.004


In [7]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "learning_rate": [0.05, 0.1, 0.2],
    "max_iter": [200, 500],
    "max_leaf_nodes": [31, 63],
}

grid = GridSearchCV(
    HistGradientBoostingRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)
grid.fit(X_train, y_train)

print("Parameter terbaik:", grid.best_params_)
print(f"CV MAE terbaik: {-grid.best_score_:.3f}")

Parameter terbaik: {'learning_rate': 0.1, 'max_iter': 200, 'max_leaf_nodes': 63}
CV MAE terbaik: 0.304
